# Introduction to Object Detection   
## CSCI E-25
### Steve Elston    

## Introduction   

Object detection is widely used computer vision task. Object detectors are used in many applications, including:    
- **Robotics:** Located and identify objects in the working environment.    
- **Autonomous vehicles:** Navigate though complex environments while avoiding collisions.
- **Medicine:** Identify areas of an medical image of particular interest.
- **Scene understanding:** Use located and identified objects to understand the semantics of an image.

In this notebook you will do the following:      
1. Using simplified example code, develop a better understanding of the key components of an object detection algorithm.
2. Learn the basics of working with the state of the art (SOTA) Yolo26 algorithm by applying it to the small, but challenging, [homeobjects-3k](https://docs.ultralytics.com/datasets/detect/homeobjects-3k/) dataset.    

> *Attribution:* Much of the code used in this notebook was adapted from code generated using Microsoft Copilot.

## Understanding Object Detection Algorithms       

In this section you will develop a better understanding of the object detection algorithms by examining example code for the key components. The three basic algorithm components are illustrated in the figure below.       
1. **Backbone network:** The backbone network creates a multi-scale feature map from an input RGB images.     
2. **Path aggregation network:** The path aggregation network (PAN) is a top-down feature pyramid network (FPN), integrating multiple scales of image features.    
3. **Multi-task head:** The multi-task object detection head operates on a grid overlaid on the feature map created by the FPN to compute three distinct outputs.
    - The probability that an object exists at a location.
    - Four dimensions of a bounding box surrounding the object.
    - Object classification for the the object in the bounding box.        


<img src="../img/ObjectDetection.png" alt="Drawing" style="width:800px; height:400px"/>
<center><b>Basic object detector architecture</b></center>  


---------------------
----------------------
> **Note:** The code shown in the cells in this section represents simplified examples. This code is in raw Markdown cells and not intended to be executable.   
-----------------------
-------------------

### Backbone network     

The code for a simplified Yolo-style backbone network is shown in code listing 1 below. The backbone is based on a pretrained fully convolutional neural network (CNN).    

<center><b>Code Block 1: Basic multi-scale backbone model for object detection</b></center>  

> **Exercise 8-1:** Examine the foregoing code listing and answer the following questions in no more than two sentences.
> 1. How many scales does the feature map created by the backbone have?
> 2. Compared to the input image, what are these scales in terms of fraction of the input image and spatial dimensions.
> 3. Which of these scale is most likely to capture the key semantics of the overall image, and which scale is most likely to capture details of the objects in the image?

> **Answers:**
> 1.     
> 2.    
> 3.    

### Path aggregation network     

Code for a simplified path aggregation network (PAN) is shown in code block 2 below. The PAN network integrates scales using both a top-down feature pyramid network (FPN) and a bottom up network.    

<center><b>Code Block 2: Simplified example code for feature pyramid network (FPN) neck for object detector</b></center>  

> **Exercise 8-2:** Examine the code for the PAN network and answer these questions.    
> 1. In three or fewer sentences, explain the what using a PAN type neck is essential in the object detection algorithm.
> 2. Examine the code for the top-down FPN. In no more than four sentences explain the process by which the P3 tensor incorporates the information from the information from the coarser (lower dimensional) scales. The smoothing $3 \times 3$ convolutional layer is applied to suppress noise so that outliers do not affect the result of this dense task.    
> 3. The upsampling of the P5 and P4 tensors uses the nearest neighbor algorithm. What type of convolutional layer could have been used instead.  
> 4. Why is no any information from any other scale added to the P5 tensor in the top-down FPN?    
> 5. Examine the code for the bottom-up PAN. In no more than four sentences explain the process by which the P5 tensor incorporates the information from the information from all scales.
> 6. In a sentence explain how the `pan_down_p3` and `pan_down_p4` layers perform down-samples.    
> 7. In a single sentence, explain why it is not necessary to operate on the P3 tensor in the bottom-up PAN.  

> **Answers:**
> 1.    
> 2.    
>    i.    
>    ii.     
> 3.     
> 4.    
> 5.     
>    i.          
>    ii.      
> 6.    
> 7.      

### Object detection head      

The code to perform the object detection task-specific head is shown in Code Block 3 below. This algorithm estimates three types of parameters.       
1. The four parameters that define the bounding box coordinates.    
2. The objectness, or a metric an object being at a location.    
3. The classes of the object in the bounding box.      

<center><b>Code Block 3: Simplified example code for Yolo-style object detector head for bounding box, objectness and object class</b></center>  

> **Exercise 8-3:** Examine the code for the `YOLOHead` object in Code Block 3 and answer the following questions.    
> 1.In one or two sentences explain why the number of variables measuring the loss is 5 + the number of categories.    
> 2. In three or fewer sentences explain why the algorithm implemented in the head is fully convolutional and dense.  
> 3. In one or two sentences explain why the algorithm shown is anchorless.     
> 4. In one or two sentences explain the importance of the loop in the `call` method does.

> **Answers:**
> 1.     
> 2.     
> 3.    
> 4.     

### Loss functions

<center><b>Code Block 4: Loss funciton code for anchor-fee object detector</b></center>  

> **Exercise 8-4:**  Examine the loss function code in Code Block 4 and answer the following questions in three or fewer sentences.
> 1. Examine the `__init__` method of the ` YoloAnchorFreeLoss` object code. In one or two sentences explain why class binary cross entropy is appropriate for object loss and how can you interpret the value computed.
> 2. In the `call` method of the ` YoloAnchorFreeLoss` object code, the total loss is returned. In three or fewer sentences, explain why would you expect some loss types to increase during model training.       

> **Answers:**
> 1.     
> 2.     

### Code to execute the object detector    

A example of code for execution of the Yolo-style object detector code is shown in Code Block 4 below.

<center><b>Code Block 5: Code to execute Yolo-style object detector</b></center>  

## Working with Yolo26    

Yolo26 was released by Ultralytics in January of 2026. Yolo26 incorporates several significant improvements both for speed of training and inference and for accuracy. In this section you will gain a bit of hands-on experience using this package for object detection. This example uses the [homeobjects-3k](https://docs.ultralytics.com/datasets/detect/homeobjects-3k/) dataset.    

### Setup to run this notebook.  

This notebook was built and tested on Google Colab Pro+ It is intended to be executed using GPU support. Total execution time of the code should be in the range of 1-2 hours.   

As a first step, import the required packages by executing the code in the cell below.  

In [ ]:
import os
import random
import shutil
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import pandas as pd



!uv pip install ultralytics
from ultralytics import YOLO

### Download the dataset

Now, we need to download and extract the `HomeObjects-3K` dataset. Execute the code in the cell below to download and unzip the images and labels.  

In [ ]:
# Download the dataset (replace with the actual download URL if different)
!wget -nc https://github.com/ultralytics/assets/releases/download/v0.0.0/homeobjects-3K.zip # Replace with actual download URL

# Unzip the dataset
!unzip -qo homeobjects-3K.zip -d home_objects_3k


# Verify the directory structure by counting files
!echo "Number of files in home_objects_3k/images/train: " $(ls -l home_objects_3k/images/train | wc -l)
!echo "Number of files in home_objects_3k/labels/train: " $(ls -l home_objects_3k/labels/train | wc -l)

### Exploring the dataset   

The code in the cell below displays a few of the images from the dataset. The images are displayed with label bounding boxes and object labels. Execute the code and examine the results.    

> Note: There is a possibility that some of the class labels are incorrect. This maybe a result of the `class` vector having errors or from human labeler errors.  

In [ ]:
def load_yolo_labels(label_path):
    """Load YOLO annotations from a .txt file."""
    boxes = []
    if not label_path.exists():
        return boxes

    with open(label_path, "r") as f:
        for line in f.readlines():
            parts = line.strip().split()
            if len(parts) == 5:
                cls, xc, yc, w, h = map(float, parts)
                boxes.append((int(cls), xc, yc, w, h))
            else:
                print(f"Warning: Skipping malformed line in {label_path}: {line.strip()}")
    return boxes


def draw_yolo_boxes(img, boxes, class_names=None, color=(255, 0, 0)):
    """Draw YOLO-format boxes on an image."""
    h, w = img.shape[:2]

    for cls, xc, yc, bw, bh in boxes:
        # Convert normalized YOLO coords to pixel coords
        x1 = int((xc - bw / 2) * w)
        y1 = int((yc - bh / 2) * h)
        x2 = int((xc + bw / 2) * w)
        y2 = int((yc + bh / 2) * h)

        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)

        label = class_names[cls] if class_names else str(cls)
        cv2.putText(img, label, (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 2.0, color, 10) # 2) # Increased font size from 0.6 to 1.2

    return img


def sample_and_display(
    images_dir,
    labels_dir,
    n=5,
    class_names=None,
    seed=42
):
    random.seed(seed)

    images_dir = Path(images_dir)
    labels_dir = Path(labels_dir)

    image_files = list(images_dir.glob("*.jpg")) + \
                  list(images_dir.glob("*.png")) + \
                  list(images_dir.glob("*.jpeg"))

    sample = random.sample(image_files, min(n, len(image_files)))

    for img_path in sample:
        label_path = labels_dir / (img_path.stem + ".txt")

        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        boxes = load_yolo_labels(label_path)
        img = draw_yolo_boxes(img, boxes, class_names)

        plt.figure(figsize=(7, 7))
        plt.imshow(img)
        plt.title(img_path.name)
        plt.axis("off")
        plt.show()


# Example usage for Home Objects 3K (train split)
sample_and_display(
    images_dir="home_objects_3k/images/train",
    labels_dir="home_objects_3k/labels/train",
    n=6,
    class_names=[
        "bed", "couch", "chair", "table", "lamp", "tv", "laptop",
        "cabinet", "window", "door", "plant", "picture", "plant"
    ]
   # ['bed','sofa','chair','table','lamp','tv','laptop','wardrobe','window','door','potted plamt','photo frame']
)

> **Note:** These images have variable lighting and contrast. Performance of the model can likely be improved with contrast and perhaps lighting adjustments. In the interest of simplicity, we will forego this preprocessing here.   

### Training the model   

The code in the cell below does the following:       
1. The pretrained Yolo26 model is instantiated.      
2. The model is trained on the dataset.    

Two optimizers were applied to the SOTA MuSGD and the more standard AdamW. Tests showed that the results in terms of evaluation metrics was nearly identical for practical purposes for both optimizers. However, the MuSGD achieved convergence in about half the number of epochs compared to AdamW. The default parameter values for MuSGD achived the best learning curves. Whereas, some experimentation was required to achieve good learning for AdamW.    

Execute the code to in the cell below to train the model.

In [ ]:
# Load pretrained model
model = YOLO("yolo26n.pt")

# Train the model on HomeObjects-3K dataset
#lr = (0.005,0.001)
#weight_decay = 0.02
#epochs = 100
#training_results = model.train(data="HomeObjects-3K.yaml",
#                               epochs=epochs, optimizer="AdamW",
#                               lr0=lr[0], lrf=lr[1],
#                               weight_decay=weight_decay,
#                               imgsz=640)

epochs = 50
training_results = model.train(data="HomeObjects-3K.yaml",
                               epochs=epochs, optimizer="MuSGD",
                     #          lr0=lr[0], lrf=lr[1],
                     #          weight_decay=weight_decay,
                               imgsz=640)

> **Exercise 8-5:** Examine the table of model performance metrics. From left to right these metrics are:     
> i. The class of the object.     
> ii. The number of images where the object class occurs.      
> iii. The total number of instances of the object class.     
> iv. The precision for the class of object.      
> v. The recall for the class of object.    
> vi. The mean average precision at 50.     
> vii. The mean average precision 50-95.      
>
> Answer the following questions in three or fewer sentences.      
> 1. What statement can you make about how severe the class imbalance is for this dataset.       
> 2. Explain the difference between mAP50 and mAP(50-95) in terms of confidence in the object identified.     
> 3. Why must mAP(50-95) always be less than or equal to mAP50?       
> 4. Identify a pair of similar looking object classes that have low recall, and more than 50 instances. Explain this outcome as well as the generally low precision.
> 5. Continuing with the foregoing example of similar looking objects, explain the low mAP50 and mAP(50-95) for this pair of object classes.   

> **Answers:**
> 1.     
> 2.     
> 3.        
> 4.    
> 5.        

After training, we can inspect the `training_results` object to visualize the loss curves for `box_loss`, `cls_loss`, and `dfl_loss`. Execute the code in the cell below to display these curves.

In [ ]:
def plot_loss_curve(metrics_df, metric, label, ax):
  ax.plot(metrics_df['epoch'], metrics_df[metric], label=label)
  ax.set_xlabel('Epoch')
  ax.set_ylabel('Loss')
  ax.set_title(metric + ' vs. epoch')
  ax.grid(True)
  ax.legend()



# The Ultralytics `model.train()` function saves detailed training metrics
# to a 'results.csv' file within its run directory.
# We need to construct the path to this file.

# The default save directory for Ultralytics is usually 'runs/detect/train' if not specified otherwise.
# The 'training_results' object from model.train() generally has a 'path' attribute
# that points to the specific run directory. If 'path' is not available or reliable,
# we can infer the default path or explicitly define it.
# From previous cell output (KF35uzSo5YV2), the save_dir was reported as /content/runs/detect/train
results_csv_path = Path('/content/runs/detect/train') / 'results.csv'

# Load the training metrics from the CSV file
try:
    metrics_df = pd.read_csv(results_csv_path)
except FileNotFoundError:
    print(f"Error: results.csv not found at {results_csv_path}. " \
          "Please ensure the model training completed successfully (cell KF35uzSo5YV2) and check the path.")
    # If the file is not found, we cannot proceed with plotting, so we can raise the error
    raise

# Plot the loss curves
_, ax = plt.subplots(1,3, figsize=(12, 4))
plot_loss_curve(metrics_df, 'train/box_loss', 'Box Loss Train', ax[0])
plot_loss_curve(metrics_df, 'val/box_loss', 'Box Loss Val', ax[0])
plot_loss_curve(metrics_df, 'train/cls_loss', 'Class Loss Train', ax[1])
plot_loss_curve(metrics_df, 'val/cls_loss', 'Class Loss Val', ax[1])
plot_loss_curve(metrics_df, 'train/dfl_loss', 'DFL Loss Train', ax[2])
plot_loss_curve(metrics_df, 'val/dfl_loss', 'DFL Loss Val', ax[2])
plt.show()

> **Exercise 8-6:** Examine these learning curves. For the most part they look reasonable. But, how can you explain that the increase in box loss and DFL loss for the first few epochs? Write your answer in three or fewer sentences.       

> **Answer:**     

### Visualizing the estimates

Now, let's create a function to visualize sampled images with both ground truth and predicted bounding boxes and class labels. Ground truth boxes will be shown in green, and predicted boxes in blue. Execute the code in the cell below and examine the results.   

In [ ]:
# Define the class names (as used in glxQsv_kdfpu)
class_names=[
        "bed", "couch", "chair", "table", "lamp", "tv", "laptop",
        "cabinet", "window", "door", "plant", "picture", "plant"
    ]

def display_predictions_with_ground_truth(
    images_dir,
    labels_dir,
    model,
    class_names,
    n=5,
    seed=42,
    gt_color=(0, 255, 0), # Green for Ground Truth (B, G, R)
    pred_color=(0, 0, 255) # Blue for Predictions (B, G, R)
):
    random.seed(seed)

    images_dir = Path(images_dir)
    labels_dir = Path(labels_dir)

    image_files = list(images_dir.glob("*.jpg")) + \
                  list(images_dir.glob("*.png")) + \
                  list(images_dir.glob("*.jpeg"))

    if not image_files:
        print(f"No image files found in {images_dir}")
        return

    sample = random.sample(image_files, min(n, len(image_files)))

    for img_path in sample:
        # Load image
        img_bgr = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # Load Ground Truth labels
        gt_label_path = labels_dir / (img_path.stem + ".txt")
        gt_boxes = load_yolo_labels(gt_label_path)

        # Make predictions
        # verbose=False to suppress output for each prediction call
        results = model(str(img_path), verbose=False, conf=0.25) # Added confidence threshold

        # Create a copy of the image to draw on
        display_img = img_rgb.copy()

        # Draw Ground Truth boxes
        # Re-using the draw_yolo_boxes function from cell glxQsv_kdfpu
        display_img = draw_yolo_boxes(display_img, gt_boxes, class_names, color=gt_color)

        # Draw Predicted boxes
        for result in results:
            if result.boxes and len(result.boxes.data) > 0:
                predicted_boxes_formatted = []
                for *xyxy, conf, cls in result.boxes.data:
                    # xyxy are in pixel coordinates (x1, y1, x2, y2)
                    x1, y1, x2, y2 = [float(v) for v in xyxy] # Ensure float for calculations

                    # Convert to normalized (xc, yc, w, h) for draw_yolo_boxes
                    xc_norm = ((x1 + x2) / 2) / w
                    yc_norm = ((y1 + y2) / 2) / h
                    w_norm = (x2 - x1) / w
                    h_norm = (y2 - y1) / h

                    predicted_boxes_formatted.append((int(cls.item()), xc_norm, yc_norm, w_norm, h_norm))

                display_img = draw_yolo_boxes(display_img, predicted_boxes_formatted, class_names, color=pred_color)

        plt.figure(figsize=(10, 10))
        plt.imshow(display_img)
        plt.title(f"Image: {img_path.name}\nGround Truth (Green), Predictions (Blue)")
        plt.axis("off")
        plt.show()

# Example usage:
# Ensure 'model' is loaded (from cell KF35uzSo5YV2) before running this.
display_predictions_with_ground_truth(
    images_dir="home_objects_3k/images/val", # Using validation set for seeing predictions on unseen data
    labels_dir="home_objects_3k/labels/val",
    model=model,
    class_names=class_names,
    n=5
)

Notice that for the most part, the bounding boxes class types agree well with the label values. At the same time, there are some obvious errors.

#### Copyright 2026, Stephen F Elston. All rights reserved.   